# GEE Pre-processing: Friction Surface Data Acquisition

This notebook pulls geospatial data from Google Earth Engine (GEE) and external
sources, processes it, and exports aligned rasters for the Alaska fuel delivery
friction surface system.

**All outputs are in EPSG:3413** (WGS 84 / NSIDC Sea Ice Polar Stereographic North),
clipped to Alaska, at **150m resolution** with consistent pixel alignment.

### Output Rasters
| File | Source | Description |
|------|--------|-------------|
| `lulc_alaska_modal.tif` | Dynamic World | Modal land cover classification (classes 0-8) |
| `slope_alaska.tif` | FabDEM | Slope in degrees |
| `dem_alaska.tif` | FabDEM | Raw elevation values |
| `permafrost_alaska.tif` | Obu et al. 2019 | Permafrost zones (0-3) |
| `roads_presence_alaska.tif` | GRIP4 | Binary road presence |
| `roads_type_alaska.tif` | AK DOT + USGS NTD | Road surface type (paved/gravel/dirt) |
| `rivers_alaska.tif` | NHD | Major/minor river classification |

### Output Vectors
| File | Source | Description |
|------|--------|-------------|
| `airports_alaska.geojson` | FAA | Airport point locations |
| `ports_alaska.geojson` | AK DOT&PF | Port locations with type |
| `facilities_alaska.geojson` | Bulk Fuel CSV | Fuel facility sites |

**References:**
- Trochim et al. (review) -- friction surface methodology
- Atkinson et al., 2005 -- slope classification
- Obu et al., 2019 -- permafrost zonation

## 1. Setup & Authentication

In [ ]:
# --- Setup ---
#pip install earthengine-api geopandas rasterio pyproj
!pip install earthengine-api geopandas rasterio pyproj

import ee
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping
import rasterio
from rasterio.features import rasterize as rio_rasterize
import zipfile

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='gee-friction-layer-processing')  # <-- Replace with your GEE project ID

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
TARGET_CRS = 'EPSG:3413'   # NSIDC Sea Ice Polar Stereographic North
TARGET_SCALE = 150          # metres
DRIVE_FOLDER = 'friction_surface_exports'  # Google Drive folder for exports

# Alaska bounding box (generous, in WGS84)
ALASKA_BBOX = ee.Geometry.Rectangle([-180, 51, -129, 72])

# Alaska state boundary from TIGER/Census for precise clipping
alaska_states = ee.FeatureCollection('TIGER/2018/States')
alaska_boundary = alaska_states.filter(ee.Filter.eq('NAME', 'Alaska')).geometry()

# Shared CRS transform for pixel alignment across all exports.
CRS_TRANSFORM = [150, 0, -3000000, 0, -150, 3000000]

print(f"Target CRS: {TARGET_CRS}")
print(f"Target scale: {TARGET_SCALE}m")
print(f"Drive folder: {DRIVE_FOLDER}")
print("GEE initialized successfully.")

In [ ]:
#Print current working directory
os.getcwd()

#Mount drive
from google.colab import drive
drive.mount('/content/drive')

#Change CWD
os.chdir('/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/notebooks/')

In [ ]:
def export_aligned_raster(image, description, filename, region=None,
                          scale=TARGET_SCALE, crs=TARGET_CRS,
                          crs_transform=CRS_TRANSFORM, folder=DRIVE_FOLDER,
                          max_pixels=1e10):
    """Export a GEE image to Google Drive with consistent alignment."""
    if region is None:
        region = alaska_boundary

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=filename,
        region=region,
        crs=crs,
        crsTransform=crs_transform,
        maxPixels=int(max_pixels),
        fileFormat='GeoTIFF',
    )
    task.start()
    print(f"Export started: {description} -> {folder}/{filename}.tif")
    return task


# Collect all tasks for monitoring
export_tasks = []

## 2. LULC -- Dynamic World (Modal Land Cover)

In [ ]:
# Dynamic World v1 -- compute per-pixel modal (most common) class over 2023
# Classes: 0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops,
#          5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice

dw_collection = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate('2023-01-01', '2023-12-31')
    .filterBounds(alaska_boundary)
    .select('label')
)

# Compute per-pixel mode (most frequent class across the year)
lulc_modal = dw_collection.mode().clip(alaska_boundary).rename('lulc')

# Export
task = export_aligned_raster(
    image=lulc_modal.toInt8(),
    description='LULC_Alaska_Modal_2023',
    filename='lulc_alaska_modal',
)
export_tasks.append(task)

print("Dynamic World LULC classes:")
print("  0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops")
print("  5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice")

## 3. DEM & Slope -- FabDEM

In [ ]:
# FabDEM -- Forest And Buildings removed Copernicus DEM
# Source: https://gee-community-catalog.org/projects/fabdem/
fabdem = ee.ImageCollection('projects/sat-io/open-datasets/FABDEM').mosaic()

dem_alaska = fabdem.clip(alaska_boundary).rename('elevation')

# Export raw DEM (elevation in metres)
task_dem = export_aligned_raster(
    image=dem_alaska.toFloat(),
    description='DEM_Alaska_FabDEM',
    filename='dem_alaska',
)
export_tasks.append(task_dem)

# Compute slope in degrees
slope_alaska = ee.Terrain.slope(dem_alaska).rename('slope')

# Export slope
task_slope = export_aligned_raster(
    image=slope_alaska.toFloat(),
    description='Slope_Alaska_FabDEM',
    filename='slope_alaska',
)
export_tasks.append(task_slope)

print("DEM and slope exports started.")

## 4. Permafrost -- Obu et al., 2019

In [ ]:
# Cell 4: Permafrost -- Obu et al. 2018 (direct from Pangaea)
# Source: https://doi.pangaea.de/10.1594/PANGAEA.888600
# Native CRS: EPSG:3995 (Arctic Polar Stereographic)

import os
import zipfile
import urllib.request
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
import geopandas as gpd

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
os.makedirs(RASTER_DIR, exist_ok=True)

# 1. Download
PERPROB_URL = (
    "https://store.pangaea.de/Publications/ObuJ-etal_2018/"
    "UiO_PEX_PERPROB_5.0_20181128_2000_2016_NH.zip"
)
zip_path    = "/content/perprob.zip"
extract_dir = "/content/perprob"

print("Downloading PERPROB (~100 MB)...")
urllib.request.urlretrieve(PERPROB_URL, zip_path)
print("Download complete.")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)
    tif_files = [f for f in z.namelist() if f.endswith('.tif')]
    print(f"Extracted: {tif_files}")

perprob_src = os.path.join(extract_dir, tif_files[0])

# 2. Reproject to EPSG:3413
reproj_path = "/content/perprob_3413.tif"
dst_crs = CRS.from_epsg(3413)

with rasterio.open(perprob_src) as src:
    print(f"Source CRS: {src.crs}  |  shape: {src.shape}  |  dtype: {src.dtypes[0]}")
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    profile = src.profile.copy()
    profile.update(crs=dst_crs, transform=transform, width=width, height=height, nodata=-9999)
    with rasterio.open(reproj_path, 'w', **profile) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )

print(f"Reprojected to EPSG:3413: {reproj_path}")

# 3. Clip to Alaska boundary
try:
    ak_geom_info = alaska_boundary.getInfo()
    ak_gdf = gpd.GeoDataFrame.from_features([{
        "type": "Feature", "geometry": ak_geom_info, "properties": {}
    }], crs="EPSG:4326").to_crs("EPSG:3413")
    print("Alaska boundary sourced from GEE.")
except Exception:
    print("GEE boundary unavailable -- downloading Census TIGER shapefile...")
    states_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip"
    urllib.request.urlretrieve(states_url, "/content/states.zip")
    with zipfile.ZipFile("/content/states.zip", "r") as z:
        z.extractall("/content/states")
    states = gpd.read_file("/content/states/cb_2018_us_state_20m.shp")
    ak_gdf = states[states["NAME"] == "Alaska"].to_crs("EPSG:3413")

clipped_path = "/content/perprob_3413_ak.tif"
ak_shapes = [mapping(geom) for geom in ak_gdf.geometry]

with rasterio.open(reproj_path) as src:
    out_image, out_transform = rio_mask(src, ak_shapes, crop=True, nodata=-9999)
    profile = src.profile.copy()
    profile.update(transform=out_transform, height=out_image.shape[1], width=out_image.shape[2], nodata=-9999)
    with rasterio.open(clipped_path, 'w', **profile) as dst:
        dst.write(out_image)

print(f"Clipped to Alaska: {clipped_path}")

# 4. Reclassify 0-1 probability to 4 integer zones
output_path = os.path.join(RASTER_DIR, 'permafrost_alaska.tif')

with rasterio.open(clipped_path) as src:
    data = src.read(1).astype(np.float32)
    nodata_mask = (data == -9999)
    zones = np.zeros(data.shape, dtype=np.int8)
    zones[(data >= 0.1) & (data < 0.5)] = 1   # sporadic
    zones[(data >= 0.5) & (data < 0.9)] = 2   # discontinuous
    zones[data >= 0.9]                   = 3   # continuous
    zones[nodata_mask]                   = -1  # nodata
    profile = src.profile.copy()
    profile.update(dtype='int8', nodata=-1, compress='lzw')
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(zones, 1)

print(f"\nPermafrost raster saved: {output_path}")
print("(At native ~1 km resolution -- run Cell 4b to resample to 150 m)")

# 5. Sanity check
with rasterio.open(output_path) as src:
    d = src.read(1)
    valid = d[d != -1]
    print(f"Shape:  {src.shape}")
    print(f"CRS:    {src.crs}")
    print(f"Res:    {src.res[0]:.0f} m")
    print(f"Valid pixels: {len(valid):,}")
    for zone, label in {0: 'none', 1: 'sporadic', 2: 'discontinuous', 3: 'continuous'}.items():
        pct = 100 * np.sum(valid == zone) / len(valid)
        print(f"  Zone {zone} ({label:>14s}): {pct:5.1f}%")

## 4b. Resample Permafrost to 150m Reference Grid

In [ ]:
# Cell 4b: Resample permafrost to 150 m reference grid
# Run AFTER lulc_alaska_modal.tif has been downloaded from Google Drive.
# Uses nearest-neighbour to keep integer zone IDs intact.

import os, numpy as np, rasterio, shutil
from rasterio.warp import reproject, Resampling

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
ref_path = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
pf_path  = os.path.join(RASTER_DIR, 'permafrost_alaska.tif')
tmp_path = os.path.join(RASTER_DIR, 'permafrost_alaska_150m.tif')

if not os.path.exists(ref_path):
    raise FileNotFoundError(f"Reference raster not found: {ref_path}\nDownload lulc_alaska_modal.tif from Google Drive first.")
if not os.path.exists(pf_path):
    raise FileNotFoundError(f"Permafrost raster not found: {pf_path}\nRun Cell 4 first.")

with rasterio.open(ref_path) as ref:
    ref_profile = ref.profile.copy()
    ref_transform, ref_crs, ref_shape = ref.transform, ref.crs, (ref.height, ref.width)
print(f"Reference grid: shape={ref_shape} res={ref.res[0]:.0f} m CRS={ref_crs}")

with rasterio.open(pf_path) as src:
    print(f"Source permafrost: shape={src.shape} res={src.res[0]:.0f} m")
    out_array = np.full(ref_shape, -1, dtype=np.int8)
    reproject(source=rasterio.band(src, 1), destination=out_array,
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=ref_transform, dst_crs=ref_crs,
              resampling=Resampling.nearest, dst_nodata=-1)

ref_profile.update(dtype='int8', count=1, nodata=-1, compress='lzw')
with rasterio.open(tmp_path, 'w', **ref_profile) as dst:
    dst.write(out_array, 1)

shutil.move(tmp_path, pf_path)
print(f"\nResampled permafrost saved: {pf_path}")

# Alignment verification
with rasterio.open(pf_path) as pf, rasterio.open(ref_path) as ref:
    ok = pf.shape == ref.shape and pf.crs == ref.crs and pf.transform == ref.transform
    print(f"Alignment: {'PASS' if ok else 'FAIL'}")

with rasterio.open(pf_path) as src:
    d = src.read(1); valid = d[d != -1]
    for zone, label in {0:'none',1:'sporadic',2:'discontinuous',3:'continuous'}.items():
        pct = 100 * np.sum(valid == zone) / len(valid)
        print(f"  Zone {zone} ({label:>14s}): {pct:5.1f}%")

## 5. Roads -- GRIP4 (Presence)

In [ ]:
# GRIP4 -- Global Roads Inventory Project (version 4)
# Source: projects/sat-io/open-datasets/GRIP4/North-America

grip4_na = ee.FeatureCollection(
    'projects/sat-io/open-datasets/GRIP4/North-America'
).filterBounds(alaska_boundary)

print(f"GRIP4 North America features in Alaska: {grip4_na.size().getInfo()}")

roads_presence = (
    ee.Image(0)
    .byte()
    .paint(grip4_na, 1)
    .clip(alaska_boundary)
    .rename('road_presence')
)

print("Output: 1 = road present, 0 = no road")

task = export_aligned_raster(
    image=roads_presence.unmask(0).toInt8(),
    description='Roads_Presence_Alaska_GRIP4',
    filename='roads_presence_alaska',
)
export_tasks.append(task)

## 6. Roads -- AK DOT Surface Type

AK DOT road data is downloaded manually from the [AK DOT GIS portal](https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/)
and placed in the working directory as `akdot_roads_alaska.shp`.

This cell loads the shapefile, classifies surface types (paved/gravel/dirt),
reprojects to EPSG:3413, and rasterizes to match the reference grid.

In [ ]:
# AK DOT Roads -- load from working directory, classify surface type, rasterize
# Source: https://gis.data.alaska.gov/datasets/AKDOT::roads-akdot/

# Extract zip file
with zipfile.ZipFile('/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/Roads_AKDOT.zip', 'r') as z:
    z.extractall('/content/akdot_roads')
print(os.listdir('/content/akdot_roads'))
AKDOT_ROADS_PATH = '/content/akdot_roads/Roads_AKDOT.shp'

if not os.path.exists(AKDOT_ROADS_PATH):
    raise FileNotFoundError(f"AK DOT roads shapefile not found at: {AKDOT_ROADS_PATH}")

roads_gdf = gpd.read_file(AKDOT_ROADS_PATH)
print(f"Loaded {len(roads_gdf)} road features from {AKDOT_ROADS_PATH}")
print(f"CRS: {roads_gdf.crs}")
print(f"Columns: {list(roads_gdf.columns)}")

# Auto-detect surface type column
surface_col = None
for col in ['SURFACE', 'SurfaceTyp', 'SURF_TYPE', 'Surface', 'surface_type',
            'SurfType', 'SURFTYPE', 'Surface_Ty']:
    if col in roads_gdf.columns:
        surface_col = col
        break

if surface_col is None:
    print("\nWARNING: Could not auto-detect surface type column.")
    print("Available columns:", list(roads_gdf.columns))
    print("Please set surface_col manually below and re-run.")
else:
    print(f"\nSurface type column: '{surface_col}'")
    print(roads_gdf[surface_col].value_counts())

# Classify surface types: paved=1, gravel=2, dirt=3
paved_keywords = ['paved', 'asphalt', 'concrete', 'bituminous', 'sealed', 'ac', 'pcc']
gravel_keywords = ['gravel', 'aggregate', 'crushed', 'gvl']

def classify_surface(val):
    if pd.isna(val):
        return 3
    val_lower = str(val).lower().strip()
    if any(k in val_lower for k in paved_keywords):
        return 1
    elif any(k in val_lower for k in gravel_keywords):
        return 2
    else:
        return 3

if surface_col is not None:
    roads_gdf = roads_gdf.copy()
    roads_gdf['friction_class'] = roads_gdf[surface_col].apply(classify_surface)
    class_labels = {1: 'paved', 2: 'gravel', 3: 'dirt'}
    for cls, label in class_labels.items():
        print(f"  {cls} ({label}): {(roads_gdf['friction_class'] == cls).sum():,} features")

roads_3413 = roads_gdf.to_crs(TARGET_CRS)
print(f"\nReprojected to {TARGET_CRS}")

def rasterize_road_types(roads_3413_gdf, reference_raster_path, output_path):
    with rasterio.open(reference_raster_path) as ref:
        out_shape = (ref.height, ref.width)
        out_transform = ref.transform
        out_crs = ref.crs
    shapes = [(geom, val) for geom, val in zip(
        roads_3413_gdf.geometry, roads_3413_gdf['friction_class']
    ) if geom is not None and not geom.is_empty]
    road_type_raster = rio_rasterize(shapes, out_shape=out_shape,
        transform=out_transform, fill=0, dtype='int8')
    profile = {'driver': 'GTiff', 'dtype': 'int8', 'width': out_shape[1],
        'height': out_shape[0], 'count': 1, 'crs': out_crs,
        'transform': out_transform, 'compress': 'lzw'}
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(road_type_raster, 1)
    print(f"Road type raster saved: {output_path}")

REFERENCE_RASTER = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
OUTPUT_ROAD_TYPE = os.path.join(RASTER_DIR, 'roads_type_alaska.tif')

if os.path.exists(REFERENCE_RASTER) and surface_col is not None:
    rasterize_road_types(roads_3413, REFERENCE_RASTER, OUTPUT_ROAD_TYPE)
else:
    if not os.path.exists(REFERENCE_RASTER):
        print(f"\nReference raster not yet available: {REFERENCE_RASTER}")
    print("Road GeoDataFrame ready. Call rasterize_road_types() after GEE exports complete.")

## 7. Rivers -- NHD Hydrography

NHD Alaska data is downloaded manually from the
[USGS National Map](https://prd-tnm.s3.amazonaws.com/index.html?prefix=StagedProducts/Hydrography/NHD/State/Shape/)
and extracted. The Alaska NHDFlowline is split into 7 parts.

In [ ]:
# Cell 7: Rivers -- NHD Flowlines (merged from Drive zip)
# The Alaska NHDFlowline is split into 7 parts (NHDFlowline_0.shp through NHDFlowline_6.shp)

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
NHD_ZIP_PATH = '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/NHD_H_Alaska_State_Shape.zip'
EXTRACT_DIR  = '/content/nhd'
REFERENCE_RASTER = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
OUTPUT_RIVERS    = os.path.join(RASTER_DIR, 'rivers_alaska.tif')
SHAPE_DIR = '/content/nhd/Shape'

print("Extracting NHDFlowline files...")
with zipfile.ZipFile(NHD_ZIP_PATH, 'r') as z:
    flowline_files = [f for f in z.namelist() if 'NHDFlowline_' in f and 'VAA' not in f]
    print(f"Found {len(flowline_files)} NHDFlowline sidecar files in zip")
    for f in flowline_files:
        z.extract(f, '/content/nhd')

shp_parts = sorted([f for f in os.listdir(SHAPE_DIR)
    if f.startswith('NHDFlowline_') and f.endswith('.shp') and 'VAA' not in f])
print(f"\nMerging {len(shp_parts)} parts: {shp_parts}")

parts = []
for shp in shp_parts:
    gdf = gpd.read_file(os.path.join(SHAPE_DIR, shp))
    print(f"  {shp}: {len(gdf):,} features")
    parts.append(gdf)

flowlines = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
print(f"\nTotal merged features: {len(flowlines):,}")
print(f"CRS: {flowlines.crs}")
print(f"Columns: {list(flowlines.columns)}")

# Classify major vs minor rivers
major_alaska_rivers = [
    'Yukon', 'Kuskokwim', 'Tanana', 'Copper', 'Susitna',
    'Nushagak', 'Kvichak', 'Stikine', 'Colville', 'Noatak',
    'Kobuk', 'Porcupine', 'Innoko', 'Koyukuk',
]

if 'StreamOrde' in flowlines.columns:
    flowlines['river_class'] = np.where(flowlines['StreamOrde'] >= 4, 1, 2)
    print("\nClassified by StreamOrde (>= 4 = major)")
elif 'gnis_name' in flowlines.columns:
    is_major = flowlines['gnis_name'].str.contains(
        '|'.join(major_alaska_rivers), case=False, na=False)
    flowlines['river_class'] = np.where(is_major, 1, 2)
    print("\nClassified by gnis_name (named major Alaska rivers)")
elif 'GNIS_Name' in flowlines.columns:
    is_major = flowlines['GNIS_Name'].str.contains(
        '|'.join(major_alaska_rivers), case=False, na=False)
    flowlines['river_class'] = np.where(is_major, 1, 2)
    print("\nClassified by GNIS_Name (named major Alaska rivers)")
else:
    flowlines['river_class'] = 2
    print("\nWARNING: No StreamOrde or gnis_name column found. All classified as minor.")

print(f"  Major navigable (1): {(flowlines['river_class'] == 1).sum():,} features")
print(f"  Minor (2):           {(flowlines['river_class'] == 2).sum():,} features")

flowlines_3413 = flowlines.to_crs('EPSG:3413')
print(f"\nReprojected to EPSG:3413")

if not os.path.exists(REFERENCE_RASTER):
    raise FileNotFoundError(f"Reference raster not found: {REFERENCE_RASTER}")

with rasterio.open(REFERENCE_RASTER) as ref:
    out_shape = (ref.height, ref.width)
    out_transform = ref.transform
    out_crs = ref.crs

shapes_minor = [(geom, 2) for geom, cls in
    zip(flowlines_3413.geometry, flowlines_3413['river_class'])
    if geom is not None and not geom.is_empty and cls == 2]
shapes_major = [(geom, 1) for geom, cls in
    zip(flowlines_3413.geometry, flowlines_3413['river_class'])
    if geom is not None and not geom.is_empty and cls == 1]

print(f"\nRasterizing {len(shapes_minor):,} minor + {len(shapes_major):,} major features...")

river_raster = rio_rasterize(shapes_minor + shapes_major,
    out_shape=out_shape, transform=out_transform, fill=0, dtype='int8')

profile = {'driver': 'GTiff', 'dtype': 'int8', 'width': out_shape[1],
    'height': out_shape[0], 'count': 1, 'crs': out_crs,
    'transform': out_transform, 'compress': 'lzw', 'nodata': 0}
with rasterio.open(OUTPUT_RIVERS, 'w', **profile) as dst:
    dst.write(river_raster, 1)

print(f"\nRiver raster saved: {OUTPUT_RIVERS}")
print(f"  Major (1) pixels:  {np.sum(river_raster == 1):,}")
print(f"  Minor (2) pixels:  {np.sum(river_raster == 2):,}")

## 8. Airports -- FAA

In [ ]:
# FAA Airport Locations
try:
    faa_airports = ee.FeatureCollection('FAA/NASR/2024/airports')
    ak_airports = faa_airports.filter(ee.Filter.eq('STATE', 'AK'))
    print(f"FAA airports in Alaska: {ak_airports.size().getInfo()}")
    airport_info = ak_airports.getInfo()
    airport_records = []
    for f in airport_info['features']:
        props = f['properties']
        coords = f['geometry']['coordinates']
        airport_records.append({
            'name': props.get('ARPT_NAME', ''), 'faa_id': props.get('ARPT_ID', ''),
            'city': props.get('CITY', ''), 'state': 'AK',
            'longitude': coords[0], 'latitude': coords[1],
        })
    airports_gdf = gpd.GeoDataFrame(airport_records,
        geometry=gpd.points_from_xy([r['longitude'] for r in airport_records],
                                     [r['latitude'] for r in airport_records]),
        crs='EPSG:4326')
except Exception as e:
    print(f"GEE FAA asset not available: {e}")
    print("Using OpenFlights as backup source...")
    openflights_cols = ['id','name','city','country','iata','icao',
        'latitude','longitude','altitude','timezone','dst','tz_database','type','source']
    airports_df = pd.read_csv(
        "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat",
        header=None, names=openflights_cols)
    ak_airports_df = airports_df[
        (airports_df['country'] == 'United States') &
        (airports_df['latitude'] >= 51) & (airports_df['latitude'] <= 72) &
        (airports_df['longitude'] >= -180) & (airports_df['longitude'] <= -129)].copy()
    airports_gdf = gpd.GeoDataFrame(ak_airports_df,
        geometry=gpd.points_from_xy(ak_airports_df.longitude, ak_airports_df.latitude),
        crs='EPSG:4326')
    print(f"OpenFlights airports in Alaska: {len(airports_gdf)}")

airports_3413 = airports_gdf.to_crs('EPSG:3413')
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
airports_path = os.path.join(output_dir, 'airports_alaska.geojson')
airports_3413.to_file(airports_path, driver='GeoJSON')
print(f"Airports saved: {airports_path} ({len(airports_3413)} airports)")

## 9. Ports -- AK DOT&PF

Port data extracted from zip on Google Drive.

In [ ]:
# AK DOT&PF Port Locations
with zipfile.ZipFile('/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/AK_Ports_and_Harbors.zip', 'r') as z:
    z.extractall('/content/ak_ports_harbors')
print(os.listdir('/content/ak_ports_harbors'))
PORTS_PATH = '/content/ak_ports_harbors/Ports_and_Harbors.shp'

if not os.path.exists(PORTS_PATH):
    raise FileNotFoundError(f"Port shapefile not found at: {PORTS_PATH}")

ports_gdf = gpd.read_file(PORTS_PATH)
print(f"Loaded {len(ports_gdf)} port features from {PORTS_PATH}")
print(f"CRS: {ports_gdf.crs}")
print(f"Columns: {list(ports_gdf.columns)}")

type_col = None
for col in ['TYPE','Type','type','PORT_TYPE','PortType','FACILITY','FacilityTy','Facility_T','Facility','CLASS','Class']:
    if col in ports_gdf.columns:
        type_col = col
        break

if type_col is not None:
    print(f"\nPort type column: '{type_col}'")
    print(ports_gdf[type_col].value_counts())
else:
    print("\nWARNING: Could not auto-detect port type column.")
    print("Available columns:", list(ports_gdf.columns))

beach_keywords = ['beach','landing','lighter','barge landing','seasonal','small boat']

def classify_port(val):
    if pd.isna(val):
        return 'port'
    val_lower = str(val).lower().strip()
    if any(k in val_lower for k in beach_keywords):
        return 'beach_landing'
    return 'port'

ports_gdf = ports_gdf.copy()
if type_col is not None:
    ports_gdf['port_class'] = ports_gdf[type_col].apply(classify_port)
else:
    ports_gdf['port_class'] = 'port'

print(f"\nFull ports: {(ports_gdf['port_class'] == 'port').sum()}")
print(f"Beach landings: {(ports_gdf['port_class'] == 'beach_landing').sum()}")

ports_3413 = ports_gdf.to_crs(TARGET_CRS)
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
ports_output = os.path.join(output_dir, 'ports_alaska.geojson')
ports_3413.to_file(ports_output, driver='GeoJSON')
print(f"\nPorts exported: {ports_output} ({len(ports_3413)} features)")

## 10. Facilities -- Bulk Fuel Sites

In [ ]:
# Bulk Fuel Facility Sites -- reproject from CSV to EPSG:3413
csv_path = '../Utilities_Bulk_Fuel_Inventory.csv'
if not os.path.exists(csv_path):
    csv_path = 'Utilities_Bulk_Fuel_Inventory.csv'

bulk_fuel = pd.read_csv(csv_path, usecols=[
    'ASTFacilityID', 'ASTFacilityLongitude', 'ASTFacilityLatitude',
    'CommunityName', 'Delivery_method'
])
bulk_fuel = bulk_fuel.dropna(subset=['ASTFacilityLongitude', 'ASTFacilityLatitude'])

facilities_gdf = gpd.GeoDataFrame(bulk_fuel,
    geometry=gpd.points_from_xy(bulk_fuel['ASTFacilityLongitude'],
                                 bulk_fuel['ASTFacilityLatitude']),
    crs='EPSG:4326')

facilities_3413 = facilities_gdf.to_crs('EPSG:3413')
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
facilities_path = os.path.join(output_dir, 'facilities_alaska.geojson')
facilities_3413.to_file(facilities_path, driver='GeoJSON')
print(f"Facilities saved: {facilities_path} ({len(facilities_3413)} sites)")

## 11. Monitor GEE Export Tasks & Alignment Verification

In [ ]:
# Monitor export tasks
import time

def monitor_tasks(tasks, poll_interval=30):
    print(f"Monitoring {len(tasks)} export tasks...")
    while True:
        statuses = {}
        for t in tasks:
            status = t.status()
            statuses[status['description']] = status['state']
        completed = sum(1 for s in statuses.values() if s == 'COMPLETED')
        failed = sum(1 for s in statuses.values() if s == 'FAILED')
        running = sum(1 for s in statuses.values() if s in ('RUNNING', 'READY'))
        print(f"\r  Completed: {completed} | Running: {running} | Failed: {failed}", end='')
        if running == 0:
            print()
            break
        time.sleep(poll_interval)
    print("\nFinal status:")
    for desc, state in statuses.items():
        print(f"  {desc}: {state}")

# Uncomment to monitor:
# monitor_tasks(export_tasks)

In [ ]:
# Alignment Verification
# Run this AFTER downloading exported rasters from Google Drive to the rasters/ directory

import os
import numpy as np
import rasterio

raster_dir = os.getenv('RASTER_DIR', './rasters')
raster_files = {
    'LULC': os.path.join(raster_dir, 'lulc_alaska_modal.tif'),
    'Slope': os.path.join(raster_dir, 'slope_alaska.tif'),
    'DEM': os.path.join(raster_dir, 'dem_alaska.tif'),
    'Permafrost': os.path.join(raster_dir, 'permafrost_alaska.tif'),
    'Roads Presence': os.path.join(raster_dir, 'roads_presence_alaska.tif'),
    'Rivers': os.path.join(raster_dir, 'rivers_alaska.tif'),
}

print(f"{'Layer':<20} {'Shape':<20} {'CRS':<15} {'Res (m)':<12} {'Dtype':<10} {'Min':<10} {'Max':<10}")
print("-" * 97)

reference_crs = reference_transform = reference_shape = None
all_aligned = True

for name, path in raster_files.items():
    if not os.path.exists(path):
        print(f"{name:<20} FILE NOT FOUND: {path}")
        all_aligned = False
        continue
    with rasterio.open(path) as src:
        data = src.read(1)
        crs, res, shape = str(src.crs), src.res, (src.height, src.width)
        dtype, transform = str(src.dtypes[0]), src.transform
        if reference_crs is None:
            reference_crs, reference_transform, reference_shape = crs, transform, shape
        else:
            if crs != reference_crs or shape != reference_shape or transform != reference_transform:
                all_aligned = False
        valid = data[data != src.nodata] if src.nodata else data
        vmin = f"{np.nanmin(valid):.1f}" if len(valid) > 0 else "N/A"
        vmax = f"{np.nanmax(valid):.1f}" if len(valid) > 0 else "N/A"
        print(f"{name:<20} {str(shape):<20} {crs:<15} {res[0]:<12.1f} {dtype:<10} {vmin:<10} {vmax:<10}")

print()
print("ALL RASTERS ALIGNED" if all_aligned else "WARNING: Raster alignment issues detected.")